# 글로벌 강수량 패턴 클러스터링
## WeatherBench ERA5 데이터 + K-Means 머신러닝 실습

---

### 실습 목표

이 노트북은 **2020년 1년간의 전 지구 강수량 데이터**를 분석하고,  
머신러닝(K-Means)으로 강수 패턴을 군집화한 뒤 세계 지도에 시각화하는 실습입니다.

| 단계 | 내용 |
|------|------|
| **Step 1** | 패키지 설치 및 임포트 |
| **Step 2** | WeatherBench 데이터 로드 |
| **Step 3** | 피처 추출 및 전처리 (연평균, 계절 편차 등) |
| **Step 4** | K-Means 클러스터링 |
| **Step 5** | 세계 지도 시각화 |
| **Step 6** | 클러스터별 패턴 해석 |

### 사용 데이터
- **WeatherBench 2** — ERA5 재분석 강수량 (`total_precipitation_6hr`)
- 기간: 2020년 전체 | 해상도: **5.625°** (위도 32 × 경도 64)
- 저장소: Google Cloud Storage 공개 버킷

---
## Step 1. 패키지 설치 및 임포트

| 패키지 | 주요 역할 |
|--------|-----------|
| `xarray` | 다차원 기후 데이터 처리 |
| `zarr` / `gcsfs` | GCS의 Zarr 형식 파일 접근 |
| `scikit-learn` | K-Means 클러스터링, 전처리 |
| `cartopy` | 지구 투영법 기반 지도 시각화 |
| `matplotlib` | 그래프 및 지도 렌더링 |

# 필요 패키지 설치 (처음 실행 시 또는 Colab 세션 시작 시)
#!pip install -q xarray zarr gcsfs scikit-learn matplotlib cartopy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/weather-climate-ai-tutorials

In [ ]:
# 나눔 고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import gcsfs

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

# ── 한글 폰트 설치 및 등록 ───────────────────────────────────────────
font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'

if not font_path.exists():
    print('NanumGothic 폰트 다운로드 중...')
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path
        )
        print('다운로드 완료')
    except Exception as e:
        print(f'다운로드 실패: {e}')

if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    _font_name = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams['font.family'] = _font_name
    print(f'폰트 등록 완료: {_font_name}')
else:
    for sys_path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
        if pathlib.Path(sys_path).exists():
            fm.fontManager.addfont(sys_path)
            plt.rcParams['font.family'] = fm.FontProperties(fname=sys_path).get_name()
            print(f'시스템 폰트 사용: {sys_path}')
            break
    else:
        print('한글 폰트 없음 → 기본 폰트 사용')

print("라이브러리 임포트 완료")
print(f"  numpy  : {np.__version__}")
print(f"  xarray : {xr.__version__}")

---
## Step 2. 데이터 로드

### WeatherBench 2 데이터셋

**WeatherBench**는 날씨 예측 모델의 공정한 비교·평가를 위한 오픈 벤치마크 데이터셋으로,  
ERA5 재분석 데이터를 Google Cloud Storage(GCS)에 공개하고 있습니다.

- **5.625° 해상도** = 위도 32개 × 경도 64개 격자 (고해상도 대비 데이터 크기 ~1/64)
- **Zarr 포맷**: 클라우드 최적화 배열 형식 — 전체 파일을 다운로드하지 않아도 필요한 부분만 읽을 수 있음
- `gcsfs`의 **익명 접근 모드**(`token='anon'`)를 사용하므로 Google 계정 인증 불필요

```
GCS 경로:
gs://weatherbench2/datasets/era5/1959-2023_01_10-6h-64x32_equiangular_with_poles_conservative.zarr
```

In [ ]:
# ── GCS 익명 파일시스템 연결 ──────────────────────────────────────────
fs = gcsfs.GCSFileSystem(token='anon')

GCS_PATH = (
    "gs://weatherbench2/datasets/era5/"
    "1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
)

print(f"GCS 경로: gs://{GCS_PATH}")
store = fs.get_mapper(GCS_PATH)

# xarray로 열기 — lazy loading: 메타데이터만 먼저 읽음
ds_full = xr.open_zarr(store, consolidated=True)
print(f"데이터셋 변수 목록: {list(ds_full.data_vars)}")

In [ ]:
# ── 2020년 강수량 데이터 선택 및 다운로드 ────────────────────────────
VAR  = 'total_precipitation_6hr'   # 6시간 누적 강수량 (단위: m)
YEAR = '2020'

# .sel()로 시간 범위 지정 (아직 lazy)
da_raw = ds_full[VAR].sel(time=YEAR)

print("데이터 정보 (로드 전):")
print(f"  차원    : {dict(da_raw.sizes)}")
print(f"  시간 범위: {str(da_raw.time.values[0])[:10]}  ~  {str(da_raw.time.values[-1])[:10]}")
print(f"  위도    : {float(da_raw.latitude.min()):.2f}° ~ {float(da_raw.latitude.max()):.2f}°")
print(f"  경도    : {float(da_raw.longitude.min()):.2f}° ~ {float(da_raw.longitude.max()):.2f}°")
print(f"\n실제 데이터 다운로드 중...")

# .load() 호출 시 GCS에서 실제 다운로드
da_raw = da_raw.load()

print(f"다운로드 완료  (메모리: {da_raw.nbytes / 1e6:.1f} MB)")

In [ ]:
# m → mm 단위 변환 + 차원 순서를 (time, latitude, longitude)으로 통일
da_precip = (da_raw * 1000).transpose('time', 'latitude', 'longitude')
da_precip.attrs['units'] = 'mm'

print(f"da_precip 차원: {dict(da_precip.sizes)}")
print(f"  shape: {da_precip.shape}  → (time, lat, lon) 순서 확인")

In [ ]:
# ── 2m 기온 데이터 로드 (Köppen 분류 + 기온 피처용) ───────────────────
TEMP_VAR = '2m_temperature'
da_temp_raw = ds_full[TEMP_VAR].sel(time=YEAR).load()

# K → ℃ 변환 + 차원 순서를 (time, latitude, longitude)으로 통일
da_temp = (da_temp_raw - 273.15).transpose('time', 'latitude', 'longitude')
da_temp.attrs['units'] = '°C'

print(f"기온 데이터 로드 완료")
print(f"  차원    : {dict(da_temp.sizes)}")
print(f"  값 범위 : {float(da_temp.min()):.1f} ~ {float(da_temp.max()):.1f} °C")

# Köppen 분류 및 피처 계산용 월별 집계
monthly_temp_mean = da_temp.resample(time='1ME').mean()   # (12, lat, lon) °C
print(f"\n월별 평균 기온 shape: {monthly_temp_mean.shape}")

---
## Step 3. 피처 추출 및 전처리

K-Means는 **각 격자점(위도-경도)을 하나의 '샘플'** 로, **통계 특성을 '피처(feature)'** 로 입력받습니다.  
2020년 1년간의 시계열에서 아래 **7가지 통계 피처**를 격자별로 추출합니다.

| 피처 번호 | 설명 | 계절 포함 여부 |
|-----------|------|---------------|
| F1 | 연평균 강수량 (mm/day) | ─ |
| F2 | 연간 강수량 표준편차 | ─ |
| F3 | 계절별 강수량 편차 — 겨울(DJF) | DJF |
| F4 | 계절별 강수량 편차 — 봄 (MAM) | MAM |
| F5 | 계절별 강수량 편차 — 여름(JJA) | JJA |
| F6 | 계절별 강수량 편차 — 가을(SON) | SON |
| F7 | 최대 계절 강수 비율 (wet season ratio) | ─ |

**계절별 강수량 편차** = 해당 계절 평균 − 연 평균  
→ 강수량이 많은 계절과 적은 계절의 차이를 포착합니다.

**최종 행렬 X 형태:**
```
X.shape = (32 × 64, 7) = (2048 샘플, 7 피처)
```

In [ ]:
# ── 피처 1 · 2: 연평균 강수량, 표준편차 ────────────────────────────────
# 6시간 간격이므로 하루 4번 → mm/day 환산
daily_mean   = da_precip.mean(dim='time') * 4      # F1: mm/day
annual_total = da_precip.sum(dim='time')            # 연간 누적 (mm/year)
daily_std    = da_precip.std(dim='time') * 4        # F2: 일 강수량 표준편차

print("F1 연평균 강수량 (mm/day):")
print(f"  min={float(daily_mean.min()):.3f}  max={float(daily_mean.max()):.3f}  mean={float(daily_mean.mean()):.3f}")

In [ ]:
# ── 피처 3~6: 계절별 강수량 편차 ──────────────────────────────────────
#
# 계절 정의 (북반구 기준)
#   DJF: 12월, 1월, 2월  (겨울)
#   MAM:  3월, 4월, 5월  (봄)
#   JJA:  6월, 7월, 8월  (여름)
#   SON:  9월, 10월, 11월 (가을)
#
# xarray의 .sel(time=da_precip.time.dt.month.isin([...])) 로 월을 선택합니다.
# ──────────────────────────────────────────────────────────────────────

seasons = {
    'DJF': [12, 1, 2],
    'MAM': [3, 4, 5],
    'JJA': [6, 7, 8],
    'SON': [9, 10, 11],
}

seasonal_mean = {}   # 계절별 평균 강수량 (mm/day)
seasonal_dev  = {}   # 계절별 편차 = 계절 평균 - 연 평균

for season_name, months in seasons.items():
    mask = da_precip.time.dt.month.isin(months)
    s_mean = da_precip.sel(time=mask).mean(dim='time') * 4   # mm/day
    s_dev  = s_mean - daily_mean                              # 편차

    seasonal_mean[season_name] = s_mean
    seasonal_dev[season_name]  = s_dev

    print(f"F{list(seasons.keys()).index(season_name)+3} {season_name} 편차:  "
          f"min={float(s_dev.min()):+.3f}  max={float(s_dev.max()):+.3f}  "
          f"mean={float(s_dev.mean()):+.4f} mm/day")

In [ ]:
# ── 피처 7: Wet Season Ratio (최강 계절 강수 비율) ─────────────────────
#
# 네 계절 중 가장 비가 많은 계절의 강수량 / 연 총 강수량
# → 1에 가까울수록 특정 계절에 강수가 집중됨 (몬순형)
# → 0.25에 가까울수록 연중 균일 (열대우림, 해양성 기후)
# ──────────────────────────────────────────────────────────────────────

# 네 계절 강수량을 쌓아서 최댓값 계산
stacked = xr.concat(list(seasonal_mean.values()), dim='season')
max_season = stacked.max(dim='season')   # 격자별 최대 계절 강수량

# wet_ratio = max_season / (연 평균 * 4) — 분모가 0이면 NaN 처리
annual_day_mean = daily_mean.where(daily_mean > 0)   # 0인 격자 → NaN
wet_ratio = max_season / (annual_day_mean * 4)        # 0~1 사이 값

print(f"F7 Wet Season Ratio:  min={float(wet_ratio.min()):.3f}  max={float(wet_ratio.max()):.3f}")

In [ ]:
# ── 2D 피처 행렬 조립 및 NaN 처리 ─────────────────────────────────────
#
# 1. 각 피처(DataArray, shape=32×64)를 numpy 배열로 변환 후 Flatten
# 2. 열(column) 방향으로 쌓아 (2048, 7) 행렬 생성
# 3. NaN 확인 → 0으로 대체 (건조 지역의 wet_ratio NaN 처리)
# ──────────────────────────────────────────────────────────────────────

feature_arrays = [
    daily_mean.values.ravel(),              # F1: 연평균 강수량
    daily_std.values.ravel(),               # F2: 표준편차
    seasonal_dev['DJF'].values.ravel(),     # F3: DJF 편차
    seasonal_dev['MAM'].values.ravel(),     # F4: MAM 편차
    seasonal_dev['JJA'].values.ravel(),     # F5: JJA 편차
    seasonal_dev['SON'].values.ravel(),     # F6: SON 편차
    wet_ratio.values.ravel(),              # F7: wet season ratio
]

feature_names = [
    'F1 연평균(mm/day)',
    'F2 표준편차',
    'F3 DJF편차',
    'F4 MAM편차',
    'F5 JJA편차',
    'F6 SON편차',
    'F7 Wet Ratio',
]

# (n_samples, n_features) 행렬
X_raw = np.column_stack(feature_arrays)

n_lat = da_precip.sizes['latitude']
n_lon = da_precip.sizes['longitude']
print(f"피처 행렬 형태: {X_raw.shape}")
print(f"  행(샘플) = {n_lat} lat × {n_lon} lon = {n_lat * n_lon}")
print(f"  열(피처) = {X_raw.shape[1]}개")

# ── NaN 처리 ────────────────────────────────────────────────────────
nan_counts = np.isnan(X_raw).sum(axis=0)
print("\n피처별 NaN 개수:")
for name, cnt in zip(feature_names, nan_counts):
    print(f"  {name}: {cnt}")

# 전략: NaN → 해당 열 평균값으로 대체 (imputation)
col_means = np.nanmean(X_raw, axis=0)
nan_mask  = np.isnan(X_raw)
X_raw[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

print(f"\nNaN 처리 후 NaN 수: {np.isnan(X_raw).sum()}")

In [ ]:
# ── 표준화 (StandardScaler) ───────────────────────────────────────────
#
# K-Means는 유클리드 거리 기반 → 피처 스케일이 클수록 영향이 커짐
# StandardScaler: 각 열의 평균=0, 표준편차=1로 변환
# ──────────────────────────────────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"표준화 전  — 전체 평균: {X_raw.mean():.4f},  전체 표준편차: {X_raw.std():.4f}")
print(f"표준화 후  — 전체 평균: {X_scaled.mean():.6f},  전체 표준편차: {X_scaled.std():.4f}")

# 피처 분포 시각화
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for i, (name, ax) in enumerate(zip(feature_names, axes)):
    ax.hist(X_scaled[:, i], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('표준화 값')
    ax.set_ylabel('격자 수')
    ax.grid(alpha=0.3)

axes[-1].set_visible(False)  # 7개 피처 → 마지막 빈 칸 숨김
plt.suptitle('표준화된 피처 분포 (K-Means 입력)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nK-Means 입력 행렬 X_scaled.shape = {X_scaled.shape}")

---
## Step 4. K-Means 클러스터링

### K-Means 알고리즘 원리

1. K개의 **중심점(centroid)** 을 임의로 초기화합니다.
2. 각 샘플을 **가장 가까운 중심점의 클러스터**에 할당합니다.
3. 각 클러스터의 **중심점을 소속 샘플들의 평균**으로 갱신합니다.
4. 중심점이 변하지 않을 때까지 2~3을 반복합니다.

### 최적 K 탐색

- **엘보우 방법**: WCSS(군집 내 분산 합)가 급격히 감소하다가 완만해지는 지점
- **실루엣 점수**: 군집이 잘 분리될수록 높음 (범위 −1 ~ 1)

> **실습 팁**: 아래에서 `K_FINAL = 6` 값을 바꾸어 결과가 어떻게 달라지는지 실험해 보세요!

In [ ]:
# ── 엘보우 & 실루엣 탐색 (K = 2 ~ 10) ────────────────────────────────
K_range         = range(2, 11)
wcss_list       = []
silhouette_list = []

print(f"{'K':>4} | {'WCSS':>12} | {'Silhouette':>12}")
print("-" * 34)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil    = silhouette_score(X_scaled, labels, sample_size=1000, random_state=42)

    wcss_list.append(km.inertia_)
    silhouette_list.append(sil)
    print(f"{k:>4} | {km.inertia_:>12.1f} | {sil:>12.4f}")

In [ ]:
# ── 탐색 결과 시각화 ──────────────────────────────────────────────────
best_k = list(K_range)[np.argmax(silhouette_list)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# 엘보우
ax1.plot(list(K_range), wcss_list, 'o-', color='steelblue', lw=2, ms=7)
ax1.set_title('엘보우 방법 (Elbow Method)', fontsize=13)
ax1.set_xlabel('클러스터 수 (K)')
ax1.set_ylabel('WCSS (군집 내 분산 합)')
ax1.set_xticks(list(K_range))
ax1.grid(alpha=0.3)

# 실루엣
bar_colors = ['tomato' if k == best_k else 'steelblue' for k in K_range]
ax2.bar(list(K_range), silhouette_list, color=bar_colors, edgecolor='white')
ax2.set_title('실루엣 점수 (Silhouette Score)', fontsize=13)
ax2.set_xlabel('클러스터 수 (K)')
ax2.set_ylabel('실루엣 점수')
ax2.set_xticks(list(K_range))
ax2.annotate(
    f'최고 K={best_k}',
    xy=(best_k, max(silhouette_list)),
    xytext=(best_k + 0.5, max(silhouette_list) * 1.05),
    fontsize=10, color='tomato',
    arrowprops=dict(arrowstyle='->', color='tomato')
)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('최적 클러스터 수(K) 탐색', fontsize=14)
plt.tight_layout()
plt.show()
print(f"실루엣 점수 최고: K = {best_k}")

In [ ]:
# ── 최종 K-Means 학습 ─────────────────────────────────────────────────
#
# K_FINAL 값을 바꾸어 결과가 어떻게 달라지는지 실험해 보세요!
# 권장 범위: 5 ~ 6
# ──────────────────────────────────────────────────────────────────────
K_FINAL = 5

# ── 코사인 위도 가중치 ─────────────────────────────────────────────────
# 지구 위에서 격자 면적은 cos(lat)에 비례 → 극지방 격자는 과소 가중
# sample_weight 적용으로 면적 대표성 확보 → 위도 기반 클러스터가 더 명확해짐
lat_weights_1d = np.cos(np.deg2rad(da_precip.latitude.values))  # (32,)
weights_2d     = np.tile(lat_weights_1d[:, np.newaxis], (1, n_lon))  # (32, 64)
sample_weights = weights_2d.ravel()   # (2048,)

km_final = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=20,       # 서로 다른 초기값으로 20회 반복, 최선 결과 선택
    max_iter=500
)
cluster_labels_1d = km_final.fit_predict(X_scaled, sample_weight=sample_weights)

print(f"K = {K_FINAL}  클러스터링 완료  (코사인 위도 가중치 적용)")
print(f"수렴 반복 횟수: {km_final.n_iter_}")
print(f"최종 WCSS    : {km_final.inertia_:.2f}")
print()

unique, counts = np.unique(cluster_labels_1d, return_counts=True)
print("클러스터별 격자 수:")
for u, c in zip(unique, counts):
    print(f"  클러스터 {u}: {c:>5}개  ({c / len(cluster_labels_1d) * 100:.1f}%)")

In [ ]:
# ── 1D 레이블 → 2D 격자 형태로 복원 (Reshape) ────────────────────────
#
# K-Means 출력: shape = (2048,)  (1차원 벡터)
# 복원 후     : shape = (32, 64) (위도 × 경도 격자)
#
cluster_map = cluster_labels_1d.reshape(n_lat, n_lon)

print(f"클러스터 레이블 복원 완료")
print(f"  1D 벡터: {cluster_labels_1d.shape}")
print(f"  2D 격자: {cluster_map.shape}  (lat={n_lat}, lon={n_lon})")

---
## Step 5. 세계 지도 시각화

Reshape된 클러스터 레이블을 **cartopy**로 세계 지도에 시각화합니다.

- 각 격자 픽셀은 해당 클러스터에 따라 고유한 색상으로 표시됩니다.
- **해안선(coastline)** 과 **국경선(borders)** 을 함께 그려 지리적 위치를 파악하기 쉽게 합니다.
- **범례(legend)** 에서 각 클러스터의 색상을 확인할 수 있습니다.

In [ ]:
# ── 색상 팔레트 ──────────────────────────────────────────────────────
# 클러스터 수(K_FINAL)에 맞게 색상 리스트 준비
PALETTE = [
    '#E63946',   # 클러스터 0: 빨강
    '#2196F3',   # 클러스터 1: 파랑
    '#4CAF50',   # 클러스터 2: 초록
    '#FF9800',   # 클러스터 3: 주황
    '#9C27B0',   # 클러스터 4: 보라
    '#00BCD4',   # 클러스터 5: 청록
    '#FF5722',   # 클러스터 6: 딥오렌지
    '#607D8B',   # 클러스터 7: 회청
]

colors = PALETTE[:K_FINAL]
cmap   = mcolors.ListedColormap(colors)
norm   = mcolors.BoundaryNorm(boundaries=np.arange(-0.5, K_FINAL), ncolors=K_FINAL)

lats = da_precip.latitude.values
lons = da_precip.longitude.values

print(f"격자 좌표 범위:  lat {lats[0]:.2f} ~ {lats[-1]:.2f}  |  lon {lons[0]:.2f} ~ {lons[-1]:.2f}")

In [ ]:
# ── 클러스터 세계 지도 ────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize=(17, 9),
    subplot_kw={'projection': ccrs.Robinson()}   # Robinson 투영법
)

# 클러스터 색상 격자 표시
ax.pcolormesh(
    lons, lats, cluster_map,
    cmap=cmap, norm=norm,
    transform=ccrs.PlateCarree(),
    shading='auto'
)

# 지리 특성 추가
ax.add_feature(cfeature.COASTLINE,  linewidth=0.9,  edgecolor='black')     # 해안선
ax.add_feature(cfeature.BORDERS,    linewidth=0.4,  edgecolor='dimgray')   # 국경선
ax.add_feature(cfeature.LAKES,      facecolor='none', edgecolor='black', linewidth=0.4)
ax.set_global()

# 위경도 격자선 + 레이블
gl = ax.gridlines(
    linewidth=0.5, color='gray', linestyle='--', alpha=0.5,
    xlocs=range(-180, 181, 60), ylocs=range(-90, 91, 30),
    draw_labels=True,
)
gl.top_labels   = False
gl.right_labels = False
gl.xlabel_style = {'size': 9, 'color': 'black'}
gl.ylabel_style = {'size': 9, 'color': 'black'}

# ── 범례 (Legend) ──────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(facecolor=colors[k], edgecolor='black', linewidth=0.5,
                   label=f'클러스터 {k}')
    for k in range(K_FINAL)
]
ax.legend(
    handles=legend_patches,
    loc='lower left',
    fontsize=11,
    title='강수 패턴 클러스터',
    title_fontsize=12,
    framealpha=0.9,
    edgecolor='gray',
)

ax.set_title(
    f'2020년 글로벌 강수 패턴 클러스터링  (K-Means, K={K_FINAL})\n'
    'WeatherBench ERA5  ·  5.625° 해상도  ·  피처: 연평균·표준편차·계절 편차·Wet Season Ratio',
    fontsize=13, pad=14
)

plt.tight_layout()
plt.savefig('precip_cluster_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("지도 저장 완료: precip_cluster_map.png")

In [ ]:
# ── 1년 총 강수량 세계 지도 ───────────────────────────────────────────
# annual_total = da_precip.sum(dim='time')  (cell-10에서 계산, 단위: mm/year)

fig, ax = plt.subplots(
    figsize=(17, 9),
    subplot_kw={'projection': ccrs.Robinson()}
)

pcm = ax.pcolormesh(
    lons, lats, annual_total.values,
    cmap='YlGnBu',
    vmin=0, vmax=3000,
    transform=ccrs.PlateCarree(),
    shading='nearest'
)

# 지리 특성
ax.add_feature(cfeature.COASTLINE, linewidth=0.9, edgecolor='black')
ax.add_feature(cfeature.BORDERS,   linewidth=0.4, edgecolor='dimgray')
ax.set_global()

# 위경도 격자선 + 레이블
gl = ax.gridlines(
    linewidth=0.5, color='gray', linestyle='--', alpha=0.5,
    xlocs=range(-180, 181, 60), ylocs=range(-90, 91, 30),
    draw_labels=True,
)
gl.top_labels   = False
gl.right_labels = False
gl.xlabel_style = {'size': 9, 'color': 'black'}
gl.ylabel_style = {'size': 9, 'color': 'black'}

# 컬러바
cbar = plt.colorbar(pcm, ax=ax, orientation='horizontal',
                    pad=0.04, shrink=0.6, aspect=40)
cbar.set_label('연간 총 강수량 (mm/year)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

ax.set_title(
    '2020년 연간 총 강수량  (WeatherBench ERA5  ·  5.625° 해상도)',
    fontsize=13, pad=14
)

plt.tight_layout()
plt.savefig('annual_precip_map.png', dpi=150, bbox_inches='tight')
plt.show()
print("지도 저장 완료: annual_precip_map.png")

---
## Step 6. 클러스터별 강수 패턴 분석 및 해석

각 클러스터에 속한 격자점들의 **월별 평균 강수량**을 비교하여  
어떤 기후 유형에 해당하는지 해석합니다.

In [ ]:
# ── 클러스터별 월별 평균 강수량 곡선 ─────────────────────────────────
# 월별 격자별 누적 강수량 계산
monthly_grid = da_precip.resample(time='1ME').sum()   # (12, 32, 64) mm
X_monthly    = monthly_grid.values.reshape(12, -1).T  # (2048, 12)

month_labels = ['1월','2월','3월','4월','5월','6월',
                '7월','8월','9월','10월','11월','12월']

# 클러스터별 월별 평균 계산
ncols = 3
nrows = (K_FINAL + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.8), sharey=False)
axes = axes.ravel()

for k in range(K_FINAL):
    ax      = axes[k]
    color   = colors[k]
    mask    = (cluster_labels_1d == k)
    profile = X_monthly[mask].mean(axis=0)    # 월별 평균 강수량

    ax.bar(range(12), profile, color=color, alpha=0.65, edgecolor='white')
    ax.plot(range(12), profile, 'o-', color=color, lw=2, ms=5, zorder=3)

    ax.set_title(f'클러스터 {k}  ({mask.sum()}개 격자)',
                 fontsize=11, color=color, fontweight='bold')
    ax.set_xticks(range(12))
    ax.set_xticklabels([m[:2] for m in month_labels], fontsize=8)
    ax.set_ylabel('강수량 (mm/월)')
    ax.grid(axis='y', alpha=0.3)

    annual = profile.sum()
    peak   = month_labels[np.argmax(profile)]
    ax.text(0.97, 0.95,
            f'연합계 {annual:.0f} mm\n최대: {peak}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

for i in range(K_FINAL, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f'클러스터별 월별 평균 강수 패턴  (K={K_FINAL})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 모든 클러스터 겹쳐 비교 ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

for k in range(K_FINAL):
    mask    = (cluster_labels_1d == k)
    profile = X_monthly[mask].mean(axis=0)
    ax.plot(range(12), profile, 'o-',
            color=colors[k], lw=2.5, ms=7,
            label=f'클러스터 {k}  (연합계: {profile.sum():.0f} mm)')

ax.set_title(f'클러스터별 월별 강수 패턴 비교  (K={K_FINAL})', fontsize=13)
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.set_ylabel('월 평균 강수량 (mm)')
ax.set_xlabel('월')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 7. Köppen-Geiger 기후 구분과 비교

ERA5 월별 기온·강수 데이터로 **Köppen-Geiger 5대 기후군(A/B/C/D/E)** 을 직접 계산하고,  
K-Means 클러스터 결과와 나란히 비교합니다.

| Köppen | 조건 | 대표 지역 |
|--------|------|----------|
| **A 열대** | 최한월 기온 ≥ 18°C | 아마존, 동남아, 콩고 |
| **B 건조** | 연 강수 < 임계값 r | 사하라, 아라비아, 고비 |
| **C 온대** | −3°C ≤ 최한월 < 18°C | 서유럽, 한국, 미국 동부 |
| **D 대륙** | 최한월 < −3°C | 러시아, 캐나다, 중앙아시아 |
| **E 한대** | 최난월 < 10°C | 그린란드, 남극, 고산 |

> **주의**: 이 분류는 2020년 단년 ERA5 데이터 기반 간이 구현입니다.  
> 공식 Köppen 구분은 30년 기후 평년값을 사용합니다.

In [ ]:
# ── Köppen-Geiger 간이 분류 계산 (ERA5 기반) ─────────────────────────
#
# 입력:
#   monthly_temp_mean : (12, lat, lon) °C  — Step 2에서 로드
#   monthly_grid      : (12, lat, lon) mm  — Step 6에서 계산
#
# Köppen 5대 기후군 (대분류만 사용)
#   A (열대)   : 최한월 기온 ≥ 18°C
#   B (건조)   : 연 강수 < 강수 임계값 r
#   C (온대)   : −3°C ≤ 최한월 < 18°C,  최난월 ≥ 10°C
#   D (대륙)   : 최한월 < −3°C,          최난월 ≥ 10°C
#   E (한대)   : 최난월 < 10°C
# ──────────────────────────────────────────────────────────────────────

T = monthly_temp_mean.values   # (12, lat, lon) °C
P = monthly_grid.values        # (12, lat, lon) mm  ← Step 6 monthly_grid 재사용

T_cold = T.min(axis=0)         # 최한월 기온
T_hot  = T.max(axis=0)         # 최난월 기온
T_ann  = T.mean(axis=0)        # 연평균 기온
P_ann  = P.sum(axis=0)         # 연 총 강수량 (mm)

# 강수 계절성: 북반구 기준 여름 반기(4~9월) 강수 비율
P_warm    = P[3:9].sum(axis=0)
warm_frac = P_warm / (P_ann + 1e-10)

# 건조 임계값 r (mm) — Köppen B군 판별 기준
r = np.where(warm_frac > 0.7,
             20 * (T_ann + 14),        # 하계 집중형
             np.where(warm_frac < 0.3,
                      20 * (T_ann + 7),   # 동계 집중형
                      20 * (T_ann + 10.5)))  # 균일형

# ── 기후군 배열 초기화 (−1 = 미분류) ────────────────────────────────
koppen = np.full_like(T_cold, -1, dtype=int)

koppen[T_hot < 10] = 4                                        # E 한대
koppen[(P_ann < r) & (T_hot >= 10) & (koppen == -1)] = 1      # B 건조
koppen[(T_cold >= 18) & (koppen == -1)] = 0                   # A 열대
koppen[(T_cold >= -3) & (T_hot >= 10) & (koppen == -1)] = 2   # C 온대
koppen[(T_cold < -3)  & (T_hot >= 10) & (koppen == -1)] = 3   # D 대륙
koppen[koppen == -1] = 3                                       # 미분류 → D로 처리

koppen_names = ['A 열대', 'B 건조', 'C 온대', 'D 대륙', 'E 한대']

unique_kop, counts_kop = np.unique(koppen, return_counts=True)
print("Köppen 간이 분류 결과:")
for u, c in zip(unique_kop, counts_kop):
    print(f"  {koppen_names[u]}: {c}개 격자  ({c / koppen.size * 100:.1f}%)")

In [ ]:
# ── 세계 지도 나란히 비교: K-Means vs Köppen ──────────────────────────
import matplotlib.gridspec as gridspec

koppen_colors = ['#228B22', '#D2B48C', '#90EE90', '#6495ED', '#E0FFFF']
# A열대=진초록, B건조=모래색, C온대=연초록, D대륙=파랑, E한대=연하늘

kop_cmap = mcolors.ListedColormap(koppen_colors)
kop_norm = mcolors.BoundaryNorm(boundaries=np.arange(-0.5, 5), ncolors=5)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.06)

# ── 왼쪽: K-Means 클러스터 ────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.Robinson())
ax1.pcolormesh(lons, lats, cluster_map, cmap=cmap, norm=norm,
               transform=ccrs.PlateCarree(), shading='auto')
ax1.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='black')
ax1.add_feature(cfeature.BORDERS,   linewidth=0.3, edgecolor='dimgray')
ax1.set_global()
ax1.gridlines(linewidth=0.4, color='white', linestyle='--', alpha=0.4)
ax1.set_title(f'K-Means 클러스터 (K={K_FINAL})\n강수 7개 + 기온 2개 + 건조도 1개 피처', fontsize=11)
patches1 = [mpatches.Patch(facecolor=colors[k], edgecolor='k', lw=0.5,
                            label=f'클러스터 {k}') for k in range(K_FINAL)]
ax1.legend(handles=patches1, loc='lower left', fontsize=9, framealpha=0.9)

# ── 오른쪽: Köppen 간이 분류 ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1], projection=ccrs.Robinson())
ax2.pcolormesh(lons, lats, koppen, cmap=kop_cmap, norm=kop_norm,
               transform=ccrs.PlateCarree(), shading='auto')
ax2.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='black')
ax2.add_feature(cfeature.BORDERS,   linewidth=0.3, edgecolor='dimgray')
ax2.set_global()
ax2.gridlines(linewidth=0.4, color='white', linestyle='--', alpha=0.4)
ax2.set_title('Köppen-Geiger 기후 구분 (ERA5 기반, 간이)\n5대 기후군', fontsize=11)
patches2 = [mpatches.Patch(facecolor=koppen_colors[i], edgecolor='k', lw=0.5,
                            label=koppen_names[i]) for i in range(5)]
ax2.legend(handles=patches2, loc='lower left', fontsize=9, framealpha=0.9)

plt.suptitle('K-Means 클러스터 vs Köppen-Geiger 기후 구분 비교', fontsize=14, y=1.01)
plt.savefig('cluster_vs_koppen.png', dpi=150, bbox_inches='tight')
plt.show()
print("비교 지도 저장: cluster_vs_koppen.png")

In [ ]:
# ── 클러스터별 Köppen 기후군 구성 비율 (혼동 행렬) ───────────────────
koppen_flat  = koppen.ravel()       # (2048,)
cluster_flat = cluster_labels_1d    # (2048,)

# 각 클러스터에서 Köppen 5대 기후군의 비율
cont_matrix = np.zeros((K_FINAL, 5), dtype=float)
for k in range(K_FINAL):
    mask = (cluster_flat == k)
    for j in range(5):
        cont_matrix[k, j] = (koppen_flat[mask] == j).sum() / mask.sum()

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(cont_matrix, aspect='auto', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='비율 (행 합계 = 1)')

ax.set_xticks(range(5))
ax.set_xticklabels(koppen_names, fontsize=11)
ax.set_yticks(range(K_FINAL))
ax.set_yticklabels([f'클러스터 {k}' for k in range(K_FINAL)], fontsize=11)
ax.set_xlabel('Köppen 기후군', fontsize=12)
ax.set_ylabel('K-Means 클러스터', fontsize=12)
ax.set_title('클러스터별 Köppen 기후군 구성 비율', fontsize=13)

# 셀 값 텍스트 표시
for i in range(K_FINAL):
    for j in range(5):
        val = cont_matrix[i, j]
        color = 'white' if val > 0.55 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('contingency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# 각 클러스터의 지배적 Köppen 기후군 출력
print("클러스터 → Köppen 기후군 대응:")
for k in range(K_FINAL):
    dom = np.argmax(cont_matrix[k])
    print(f"  클러스터 {k}  →  {koppen_names[dom]}  ({cont_matrix[k, dom]*100:.0f}%)")

In [ ]:
# ── 클러스터별 피처 평균값 요약표 ─────────────────────────────────────
print("클러스터별 주요 피처 평균 (원본 단위):")
print("-" * 70)
header = f"{'클러스터':>6}" + "".join(f"{n:>14}" for n in feature_names)
print(header)
print("-" * 70)

for k in range(K_FINAL):
    mask = (cluster_labels_1d == k)
    vals = X_raw[mask].mean(axis=0)
    row  = f"{k:>6}" + "".join(f"{v:>14.3f}" for v in vals)
    print(row)

print("-" * 70)

---
## 결과 해석 가이드

위의 요약표와 지도를 참고해 각 클러스터의 기후 특성을 분류해 보세요.

| 패턴 특징 | 대표 기후 | 예상 지역 |
|-----------|----------|-----------|
| 연 강수 극히 적음, 계절 편차 ≈ 0 | 사막/극건조 | 사하라, 아라비아 반도, 중앙아시아 |
| 연 강수 많음, 계절 변화 균일 | 열대우림(Af) | 아마존, 콩고 분지, 동남아시아 도서 |
| 여름(JJA) 편차 크게 양수 (북반구) | 몬순/대륙성 | 인도, 동아시아, 서아프리카 |
| 여름(DJF → 남반구) 편차 크게 양수 | 남반구 몬순 | 남미 북부, 남아프리카 |
| 겨울(DJF) 편차 양수, 여름 음수 | 지중해성(Cs) | 지중해 연안, 캘리포니아, 칠레 중부 |
| 연 강수 중간, 계절 편차 작음 | 해양성/온대 | 서유럽, 뉴질랜드 서해안 |

---

## 토론 질문

1. `K_FINAL`을 5 또는 7로 바꾸면 클러스터 경계가 어떻게 달라지나요?
2. 표준화를 생략하면 강수량이 많은 열대 지역 위주로 클러스터가 구성되는 경향이 있습니다. 직접 확인해 보세요.
3. **위도** 값을 추가 피처로 넣으면 클러스터가 어떻게 바뀔까요?
4. K-Means 대신 **DBSCAN**이나 **계층적 클러스터링**을 사용하면 어떤 차이가 있을까요?
5. 2010년 데이터로 같은 분석을 반복해 기후 변화 패턴을 비교해 보세요.